# Spatial Clusters: Getis-Ord Gi*

Are high nitrate values spread randomly, or do they form clusters? The **Getis-Ord Gi\*** statistic compares each station (and its neighbours) with the overall average:

- a high positive z-score means the station is surrounded by high values: **hot spot**
- a high negative z-score means it is surrounded by low values: **cold spot**

**Settings**
- Variable: mean nitrate per station (2012-2022), transformed with log10(x + 0.1) because the data are strongly skewed
- Neighbourhood: the 10 nearest stations (including the station itself), distance measured on the sphere (haversine)
- Confidence levels: 95% (|z| >= 1.96) and 99% (|z| >= 2.58)
- No correction for multiple testing, so the result is read as **exploratory**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.neighbors import BallTree

In [ ]:
df = pd.read_csv("../data/processed/powerbi_groundwater_nitrate.csv")

stations = df.groupby("StationCode").agg(
    FederalState=("FederalState", "first"),
    Longitude=("Longitude", "first"),
    Latitude=("Latitude", "first"),
    MeanNitrate=("Nitrate_mg_L", "mean"),
).reset_index()

print(stations.shape)
stations.head()

## 10 nearest neighbours

In [ ]:
coords = np.radians(stations[["Latitude", "Longitude"]].values)

tree = BallTree(coords, metric="haversine")

K = 10
# the nearest neighbour of a point is the point itself, so k=10 includes the station
distances, neighbours = tree.query(coords, k=K)

neighbours.shape

## Gi* z-score

For each station *i*, with *n* stations, binary weights *w* (1 for the 10 nearest, including itself), mean x̄ and standard deviation S:

z = (Σ w·x − x̄ · Σ w) / (S · √((n · Σ w² − (Σ w)²) / (n − 1)))

In [ ]:
z = np.log10(stations["MeanNitrate"].values + 0.1)

n = len(z)
x_bar = z.mean()
S = np.sqrt((z ** 2).mean() - x_bar ** 2)

numerator = z[neighbours].sum(axis=1) - x_bar * K
denominator = S * np.sqrt((n * K - K ** 2) / (n - 1))

stations["GiStar_z"] = numerator / denominator

stations["GiStar_z"].describe()

In [ ]:
def classify(z_value):
    if z_value >= 2.58:
        return "Hot spot 99%"
    if z_value >= 1.96:
        return "Hot spot 95%"
    if z_value <= -2.58:
        return "Cold spot 99%"
    if z_value <= -1.96:
        return "Cold spot 95%"
    return "Not significant"


stations["GiClass"] = stations["GiStar_z"].map(classify)

stations["GiClass"].value_counts()

In [ ]:
hot = stations["GiClass"].str.startswith("Hot").sum()
cold = stations["GiClass"].str.startswith("Cold").sum()

print("Hot spots:", hot)
print("Cold spots:", cold)

print("\nHot spots by federal state:")
print(stations[stations["GiClass"].str.startswith("Hot")]["FederalState"].value_counts().head(5).to_string())

print("\nCold spots by federal state:")
print(stations[stations["GiClass"].str.startswith("Cold")]["FederalState"].value_counts().head(5).to_string())

**Result:** 52 stations are hot spots, mainly in Sachsen and Rheinland-Pfalz. 71 are cold spots, mostly in Brandenburg and Mecklenburg-Vorpommern.

In [ ]:
colors = {
    "Hot spot 99%": "#b2182b",
    "Hot spot 95%": "#ef8a62",
    "Not significant": "#d9d9d4",
    "Cold spot 95%": "#67a9cf",
    "Cold spot 99%": "#2166ac",
}

plt.figure(figsize=(6, 8))

for cls, color in colors.items():
    subset = stations[stations["GiClass"] == cls]
    plt.scatter(subset["Longitude"], subset["Latitude"],
                s=18, color=color, label=f"{cls} ({len(subset)})")

plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Getis-Ord Gi* clusters of mean nitrate")
plt.legend(loc="lower left", fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
stations.to_csv("../data/processed/station_gi_star.csv", index=False)